In [2]:

from pyspark.sql import SparkSession

DEFAULT_SPARK_REMOTE = "sc://localhost:15002"
spark = SparkSession.builder.remote(DEFAULT_SPARK_REMOTE).getOrCreate()

spark 

In [3]:

schema = "id integer, name string"

df1 = spark.createDataFrame([(1, "Ivanov"),
                             (2, "Petrov"),
                             (3, "Sidorov")], schema)

df2 = spark.createDataFrame([(4, "Pupkin"),
                             (3, "Sidorov"),
                             (3, "Sidorov")
                             ], schema)

In [22]:
df1 = df1.withColumnRenamed("id", "df1_id").withColumnRenamed("name", "df1_name")
df2 = df2.withColumnRenamed("id", "df2_id").withColumnRenamed("name", "df2_name")

df1.show()
df2.show()

print("inner_Join (inner). Keep rows with keys that exists in the left and right datasets")
df1.join(df2, df1.df1_id == df2.df2_id).show()

print("left_outer (left, leftouter) join. Keep all rows from the left dataset and matching rows from the right dataset")
df1.join(df2, df1.df1_id == df2.df2_id, how="left_outer").show()

print("right_outer (right, rightouter) join. Keep all rows from the right dataset and matching rows from the left dataset")
df1.join(df2, df1.df1_id == df2.df2_id, how="right_outer").show()

print("""left_Semi join (semi). Keep the rows in the left dataset that have a match in the right dataset, and only the left dataset columns. 
Duplicate rows in the right dataset do not affect the result""")
df1.join(df2, df1.df1_id == df2.df2_id, how="left_semi").show()

print("left_anti join (anti). Keep the rows in the left dataset that do not have a match in the right dataset")
df1.join(df2, df1.df1_id == df2.df2_id, how="left_anti").show()

print("full_outer (full, outer, full_outer, fullouter) join. Keep all rows from both datasets, matching where possible. Synoymous ")
df1.join(df2, df1.df1_id == df2.df2_id, how="full_outer").show()

print("""cross join (or Caretesian join). Keep all rows from the left dataset combined with all rows from the right dataset.""")
df1.join(df2, how="cross").show()

print("""natural joins. Keep rows with matching keys in both datasets and combine columns with the same name.
left, right and outer natural joins are supported""")

df1.withColumnRenamed("df1_id", "id").createOrReplaceTempView("df1")
df2.withColumnRenamed("df2_id", "id").createOrReplaceTempView("df2")

spark.sql("""
          SELECT *
          FROM df1
          NATURAL INNER JOIN df2
        """).show()





+------+--------+
|df1_id|df1_name|
+------+--------+
|     1|  Ivanov|
|     2|  Petrov|
|     3| Sidorov|
+------+--------+

+------+--------+
|df2_id|df2_name|
+------+--------+
|     4|  Pupkin|
|     3| Sidorov|
|     3| Sidorov|
+------+--------+

inner_Join (inner). Keep rows with keys that exists in the left and right datasets
+------+--------+------+--------+
|df1_id|df1_name|df2_id|df2_name|
+------+--------+------+--------+
|     3| Sidorov|     3| Sidorov|
|     3| Sidorov|     3| Sidorov|
+------+--------+------+--------+

left_outer (left, leftouter) join. Keep all rows from the left dataset and matching rows from the right dataset
+------+--------+------+--------+
|df1_id|df1_name|df2_id|df2_name|
+------+--------+------+--------+
|     1|  Ivanov|  NULL|    NULL|
|     2|  Petrov|  NULL|    NULL|
|     3| Sidorov|     3| Sidorov|
|     3| Sidorov|     3| Sidorov|
+------+--------+------+--------+

right_outer (right, rightouter) join. Keep all rows from the right datase

In [8]:
from pyspark.sql.functions import expr

print("Inner Join in multiple columns")
df1.join(df2, [df1.df1_id == df2.df2_id,   
               df1.df1_name == df2.df2_name
               ]).show()

df1.alias("t1").join(df2.alias("t2"), expr("t1.df1_id = t2.df2_id and t1.df1_name = t2.df2_name")).show()


Inner Join in multiple columns
+------+--------+------+--------+
|df1_id|df1_name|df2_id|df2_name|
+------+--------+------+--------+
|     3| Sidorov|     3| Sidorov|
|     3| Sidorov|     3| Sidorov|
+------+--------+------+--------+

+------+--------+------+--------+
|df1_id|df1_name|df2_id|df2_name|
+------+--------+------+--------+
|     3| Sidorov|     3| Sidorov|
|     3| Sidorov|     3| Sidorov|
+------+--------+------+--------+

